<a href="https://colab.research.google.com/github/watch-duty/radio-transcription/blob/main/model/colabs/evaluate_transcriptions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Colab to compute metrics from a JSON manifest with transcriptions from various models.

The google drive version of this colab (for iterating and editing) [is here](https://colab.research.google.com/drive/1HJs9lUMsg5dyK4uwuLmzxzbtKEE9tkbi?authuser=1#scrollTo=72af-Ue2_BwC). Once you have a unit of edit completed, you can check-in the drive version into [this github directory](https://github.com/watch-duty/radio-transcription/tree/main/model/colabs).

In [ ]:
#@title Imports
import json
import os
from nemo.collections.asr.parts.utils import transcribe_utils
from nemo.collections.asr.metrics.wer import word_error_rate

inference_results = "/tmp/playground_parakeet_and_canary_flash.json"  #@param
postfixes_to_eval = ['parakeet-tdt-06b-v2', 'canary-1b-flash']  #@param
eval_output = "/tmp/playground_parakeet_and_canary_flash_eval.json"

In [ ]:
#@title Helper function

def run_evaluation(input_json, output_json, postfixes_to_evaluate):

  gt_text_field = "text"
  prediction_field_prefix = "pred_text_"
  pc_processor = transcribe_utils.PunctuationCapitalization(".,?")

  with open(input_json, 'r', encoding='utf-8') as f:
    with open(output_json, 'w', encoding='utf-8') as out:
      # Iterate through each utterance output in the json
      for line in f:
        data_row = json.loads(line)
        audio_filepath = data_row["audio_filepath"]
        ground_truth_text = [data_row.get(gt_text_field, "")]
        if not ground_truth_text:
          print(f"Skipping example: {audio_filepath}, ground truth is empty")
          continue
        ground_truth_text = pc_processor.do_lowercase(ground_truth_text)
        ground_truth_text = pc_processor.rm_punctuation(ground_truth_text)

        output_row = data_row
        for postfix in postfixes_to_eval:
          pred_text_field = prediction_field_prefix + postfix
          predicted_text = [data_row.get(pred_text_field)]
          if predicted_text is None:
            raise ValueError(f"Expected field: {pred_text_field} to be populated for example: {audio_filepath}")
          predicted_text = pc_processor.do_lowercase(predicted_text)
          predicted_text = pc_processor.rm_punctuation(predicted_text)

          sample_wer = word_error_rate(hypotheses=predicted_text, references=ground_truth_text, use_cer=False)
          sample_wer = round(100 * sample_wer, 2)
          output_row["wer_" + postfix] = sample_wer

        # Write as a JSON line
        out.write(json.dumps(output_row) + '\n')

  print(f"Successfully evaluated {input_json} and wrote evaluation output to {output_json}")

In [ ]:
run_evaluation(inference_results, eval_output, postfixes_to_eval)

Successfully evaluated /tmp/playground_parakeet_and_canary_flash.json and wrote evaluation output to /tmp/playground_parakeet_and_canary_flash_eval.json
